In [1]:
import os
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from PIL import Image
import torchvision.transforms as transforms

# Device selection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Placeholder transforms (identity) – can be replaced with real implementations if needed
def trim():
    return transforms.Lambda(lambda x: x)


def cropTo4_3():
    return transforms.Lambda(lambda x: x)




/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
threshold = [0.75, 1.5, 2.5, 3.5]


def regress2class(out):
    prediction = 0
    for i in range(4):
        prediction += (out.data >= threshold[i]).squeeze().cpu().item()
    return prediction


def ordinal2class_prob(out):
    pred_prob = torch.zeros(out.size(0), 5, device=out.device)
    pred_prob[:, 0] = (1 - out[:, 0]).squeeze()
    pred_prob[:, 1] = (out[:, 0] * (1 - out[:, 1])).squeeze()
    pred_prob[:, 2] = (out[:, 1] * (1 - out[:, 2])).squeeze()
    pred_prob[:, 3] = (out[:, 2] * (1 - out[:, 3])).squeeze()
    pred_prob[:, 4] = out[:, 3].squeeze()
    return F.softmax(pred_prob, dim=1)


def regress2class_prob(out):
    out = out.squeeze(-1)  # keep batch dim
    out = torch.clamp(out, 0.0, 4.0)
    pred_prob = torch.zeros((out.size(0), 5), device=out.device)
    for i in range(out.size(0)):
        val = out[i].item()
        l1 = int(math.floor(val))
        l2 = int(math.ceil(val))
        if l1 == l2:
            pred_prob[i][l1] = 1.0
        else:
            pred_prob[i][l1] = 1 - (val - l1)
            pred_prob[i][l2] = 1 - (l2 - val)
    return pred_prob


def expected_class_from_prob(prob):
    """
    Convert a probability distribution over the 5 classes (0‑4) into a single
    integer class by computing the expected value and rounding.
    """
    class_indices = torch.arange(5, device=prob.device, dtype=prob.dtype)
    exp_val = (prob * class_indices).sum(dim=1)
    return torch.round(exp_val).long()


def combine3output(r_out, c_out, o_out):
    """
    Convert each head's raw output to a probability distribution over the 5 classes,
    average the three distributions, and return the class obtained from the expected value.
    """
    prob_r = regress2class_prob(r_out)  # (B,5)
    prob_c = F.softmax(c_out, dim=1)  # (B,5)
    prob_o = ordinal2class_prob(o_out)  # (B,5)
    avg_prob = (prob_r + prob_c + prob_o) / 3.0
    pred_class_tensor = expected_class_from_prob(avg_prob)  # (B,)
    return int(pred_class_tensor.item())




In [3]:
class ThreeStage_Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            "tf_efficientnet_b4", pretrained=True, num_classes=0
        )
        feat_dim = self.backbone.num_features  # typically 1792 for B4

        self.cls_head = nn.Linear(feat_dim, 5)  # classification logits
        self.reg_head = nn.Linear(feat_dim, 1)  # regression scalar
        self.ord_head = nn.Linear(feat_dim, 4)  # ordinal logits

    def forward(self, x):
        feats = self.backbone.forward_features(x)  # (B, C, H, W)
        feats = self.backbone.global_pool(feats)  # (B, C)
        c_out = self.cls_head(feats)  # (B,5)
        r_out = self.reg_head(feats)  # (B,1)
        o_out = self.ord_head(feats)  # (B,4)
        return c_out, r_out, o_out


# Locate test CSV
test_csv_paths = [
    "../input/aptos2019-blindness-detection/test.csv",
    "./input/aptos2019-blindness-detection/test.csv",
    "../input/test.csv",
    "./input/test.csv",
]
test_ids = None
for p in test_csv_paths:
    if os.path.isfile(p):
        test_ids = pd.read_csv(p)
        break
if test_ids is None:
    raise FileNotFoundError("Test CSV file not found in expected locations.")
test_ids = test_ids["id_code"].astype(str).tolist()

# Transform pipeline
transform1 = transforms.Compose(
    [
        trim(),
        cropTo4_3(),
        transforms.Resize((288, 384)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.384, 0.258, 0.174], std=[0.124, 0.089, 0.094]),
    ]
)

# Initialise model
net1 = ThreeStage_Model()

weight_path_candidates = [
    "../input/weights/B4_3stage_22epoch_320.pkl",
    "/kaggle/input/weights/B4_3stage_22epoch_320.pkl",
    "./input/aptos2019-blindness-detection/weights/B4_3stage_22epoch_320.pkl",
    "./input/weights/B4_3stage_22epoch_320.pkl",
    "./weights/B4_3stage_22epoch_320.pkl",
    "./working/weights/B4_3stage_22epoch_320.pkl",
    "./aptos2019-blindness-detection/weights/B4_3stage_22epoch_320.pkl",
]

loaded = False
for wp in weight_path_candidates:
    if os.path.isfile(wp):
        try:
            state = torch.load(wp, map_location=device)
            net1.load_state_dict(state)
            print(f"Loaded weights from {wp}")
            loaded = True
            break
        except Exception as e:
            print(f"Failed to load weights from {wp}: {e}")

use_three_stage = loaded

if not loaded:
    print("Custom weights not found – switching to pretrained EfficientNet classifier.")
    net1 = timm.create_model("tf_efficientnet_b4", pretrained=True, num_classes=5)
    net1 = net1.to(device)
    net1.eval()
else:
    net1 = net1.to(device)
    net1.eval()

fallback_class = None
if not loaded:
    train_csv_paths = [
        "../input/aptos2019-blindness-detection/train.csv",
        "./input/aptos2019-blindness-detection/train.csv",
        "../input/train.csv",
        "./input/train.csv",
    ]
    train_df = None
    for p in train_csv_paths:
        if os.path.isfile(p):
            train_df = pd.read_csv(p)
            break
    if train_df is not None:
        fallback_class = int(train_df["diagnosis"].mode()[0])
        print(f"Fallback class (most common in training): {fallback_class}")
    else:
        fallback_class = 0
        print("Training file not found; defaulting fallback class to 0.")



model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

Custom weights not found – switching to pretrained EfficientNet classifier.
Fallback class (most common in training): 0


In [4]:
submission = []
with torch.no_grad():
    for i, idx in enumerate(test_ids):
        if i % 100 == 0:
            print(f"Processing {i}/{len(test_ids)}")
        image_path = f"../input/aptos2019-blindness-detection/test_images/{idx}.png"
        if not os.path.isfile(image_path):
            image_path = (
                f"/kaggle/input/aptos2019-blindness-detection/test_images/{idx}.png"
            )
        if not os.path.isfile(image_path):
            pred_class = fallback_class
        else:
            img = Image.open(image_path).convert("RGB")
            img = transform1(img).unsqueeze(0).to(device)
            if use_three_stage:
                c_out, r_out, o_out = net1(img)  # (cls, rg, ord)
                pred_class = combine3output(r_out, c_out, o_out)
            else:
                logits = net1(img)  # shape (1,5)
                prob = F.softmax(logits, dim=1)  # (1,5)
                pred_class = int(expected_class_from_prob(prob).cpu().item())
        submission.append([idx, pred_class])

submission = np.array(submission)



Processing 0/367
Processing 100/367
Processing 200/367
Processing 300/367


In [5]:
if len(submission) == 0:
    raise RuntimeError("Submission array is empty; inference may have failed.")
df = pd.DataFrame(submission, columns=["id_code", "diagnosis"])
output_path = "submission.csv"
df.to_csv(output_path, index=False)
print(f"Submission saved to {output_path}, rows: {len(df)}")

Submission saved to submission.csv, rows: 367
